In [ ]:
"""
Advanced smolagents + oMLX Orchestrator
=======================================
Features:
- Custom Persona & System Prompt injection
- Local file IO and structured data evaluation tools
- Sandboxed Python environment with custom library access
"""

import json
import os
from pathlib import Path
from smolagents import CodeAgent, OpenAIServerModel, tool

# 1. Local Engine Setup (oMLX)
OMLX_API_KEY = ""
OMLX_BASE_URL = "http://localhost:8000/v1" 
MODEL_ID = "Qwen3.5-9B-Claude-4.6-HighIQ-INSTRUCT-HERETIC-UNCENSORED-mlx-4Bit"

model = OpenAIServerModel(
    model_id=MODEL_ID,
    api_base=OMLX_BASE_URL,
    api_key=OMLX_API_KEY,
    temperature=0.1,  # Low temperature for strict logic and code execution
)

In [2]:
# 2. Rich Local Tool Suite
@tool
def read_local_file(file_path: str, max_chars: int = 4000) -> str:
    """Reads the contents of a local file safely from the filesystem.

    Args:
        file_path: Relative or absolute path to the file.
        max_chars: Maximum character limit to prevent context window overflow.

    Returns:
        str: Text content of the file or an error message.
    """
    path = Path(file_path)
    if not path.exists():
        return f"Error: File '{file_path}' does not exist."
    if not path.is_file():
        return f"Error: '{file_path}' is a directory, not a file."
    
    try:
        content = path.read_text(encoding="utf-8", errors="replace")
        if len(content) > max_chars:
            return content[:max_chars] + f"\n... [Truncated {len(content) - max_chars} characters]"
        return content
    except Exception as exc:
        return f"File read error: {str(exc)}"
    
@tool
def write_local_file(file_path: str, content: str) -> str:
    """Writes or overwrites text content to a local file.

    Args:
        file_path: Path where the file should be written.
        content: The text/data content to write into the file.

    Returns:
        str: Confirmation message detailing bytes written.
    """
    try:
        path = Path(file_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")
        return f"Success: Wrote {len(content.encode('utf-8'))} bytes to '{file_path}'."
    except Exception as exc:
        return f"File write error: {str(exc)}"


@tool
def run_deterministic_stat_check(records_json: str, metric_key: str) -> str:
    """Parses a JSON array of objects and calculates descriptive statistics for a specified metric.

    Args:
        records_json: A stringified JSON array containing dictionaries.
        metric_key: The numeric field name to extract and analyze.

    Returns:
        str: JSON-formatted string with min, max, mean, count, and sum.
    """
    try:
        data = json.loads(records_json)
        if not isinstance(data, list):
            return "Error: records_json must be a JSON array of objects."
        
        values = [float(row[metric_key]) for row in data if metric_key in row and row[metric_key] is not None]
        if not values:
            return f"Error: No valid numeric data found for key '{metric_key}'."
        
        summary = {
            "count": len(values),
            "sum": round(sum(values), 4),
            "mean": round(sum(values) / len(values), 4),
            "min": round(min(values), 4),
            "max": round(max(values), 4),
        }
        return json.dumps(summary, indent=2)
    except Exception as exc:
        return f"Analysis error: {str(exc)}"

In [3]:
# 3. Agent Factory with Prompt Customization
def create_specialized_agent(role_instructions: str) -> CodeAgent:
    agent = CodeAgent(
        tools=[read_local_file, write_local_file, run_deterministic_stat_check],
        model=model,
        add_base_tools=False,
        additional_authorized_imports=[
            "math",
            "json",
            "statistics",
            "collections",
            "re",
            "pathlib",
        ],
        max_steps=6,
        verbosity_level=0,
    )

    # Append custom role guidelines to smolagents' internal system prompt
    agent.prompt_templates["system_prompt"] += f"\n\n### SPECIAL OPERATIONAL GUIDELINES\n{role_instructions}\n"
    return agent

In [4]:
data_scientist_guidelines = """
You are a Principal Data Scientist and Automation Engineer.
- You write modular, clean Python algorithms to generate and analyze data.
- Validate distributions using `run_deterministic_stat_check` before concluding.
- Always provide precise numerical summaries (mean, min, max, count).
"""

ds_agent = create_specialized_agent(data_scientist_guidelines)

task_2 = """
Simulate a latency test benchmark:
1. Write a Python snippet that generates 20 synthetic latency samples (in ms) using a baseline of 45.0 ms with slight variations.
2. Structure the data as a list of dicts: [{"request_id": i, "latency_ms": value}, ...].
3. Run the 'run_deterministic_stat_check' tool on this JSON payload targeting 'latency_ms'.
4. Produce a concise final report summarizing the latency performance.
"""

print("\n--- Running Use Case 2: Data Pipeline & Validation ---")
result_2 = ds_agent.run(task_2)
print(result_2)


--- Running Use Case 2: Data Pipeline & Validation ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Simulate a latency test benchmark:                                                                              │
│ 1. Write a Python snippet that generates 20 synthetic latency samples (in ms) using a baseline of 45.0 ms with  │
│ slight variations.                                                                                              │
│ 2. Structure the data as a list of dicts: [{"request_id": i, "latency_ms": value}, ...].                        │
│ 3. Run the 'run_deterministic_stat_check' tool on this JSON payload targeting 'latency_ms'.                     │
│ 4. Produce a concise final report summarizing the latency performance.                                          │
│                                                                                                                 │
╰─ OpenAIModel - Qwen3.5-9B-Claude-4.6-HighIQ-INSTRUCT-HERETIC-UNCENSORED-mlx-4Bit ───────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import random                                                                                                    
  import math                                                                                                      
                                                                                                                   
  # Configuration                                                                                                  
  NUM_SAMPLES = 20                                                                                                 
  BASELINE_LATENCY = 45.0                                                                                          
  STDEV = 2.0  # Small variation for realistic latency                                                             
                                                                                                                   
  def generate_latency_sample(base: float, std: float, rng: random.Random) -> float:                               
      """Generate a latency sample using Box-Muller transform for normal distribution."""                          
      # Box-Muller transform for Gaussian distribution                                                             
      u1 = rng.random()                                                                                            
      u2 = rng.random()                                                                                            
      z = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)                                            
      return base + z * std                                                                                        
                                                                                                                   
  # Generate 20 synthetic latency samples                                                                          
  samples = []                                                                                                     
  for i in range(NUM_SAMPLES):                                                                                     
      latency = generate_latency_sample(BASELINE_LATENCY, STDEV, random.Random(i))                                 
      samples.append({                                                                                             
          "request_id": i,                                                                                         
          "latency_ms": round(latency, 3)  # Round to 3 decimal places for consistency                             
      })                                                                                                           
                                                                                                                   
  # Print generated data for verification                                                                          
  import json                                                                                                      
  data_json = json.dumps(samples, indent=2)                                                                        
  print("=" * 60)                                                                                                  
  print("GENERATED LATENCY DATA (20 samples):")                                                                    
  print("=" * 60)                                                                                                  
  print(data_json)                                                                                                 
  print("=" * 60)                                                                                                  
                                                         

Execution logs:
============================================================
GENERATED LATENCY DATA (20 samples):
============================================================
[
  {
    "request_id": 0,
    "latency_ms": 45.058
  },
  {
    "request_id": 1,
    "latency_ms": 47.303
  },
  {
    "request_id": 2,
    "latency_ms": 45.568
  },
  {
    "request_id": 3,
    "latency_ms": 41.741
  },
  {
    "request_id": 4,
    "latency_ms": 47.709
  },
  {
    "request_id": 5,
    "latency_ms": 44.9
  },
  {
    "request_id": 6,
    "latency_ms": 45.595
  },
  {
    "request_id": 7,
    "latency_ms": 46.752
  },
  {
    "request_id": 8,
    "latency_ms": 48.349
  },
  {
    "request_id": 9,
    "latency_ms": 43.264
  },
  {
    "request_id": 10,
    "latency_ms": 43.092
  },
  {
    "request_id": 11,
    "latency_ms": 42.656
  },
  {
    "request_id": 12,
    "latency_ms": 43.659
  },
  {
    "request_id": 13,
    "latency_ms": 43.699
  },
  {
    "request_id": 14,
    "latency_ms": 43.758
  },
  {
    "request_id": 15,
    "latency_ms": 45.531
  },
  {
    "request_id": 16,
    "latency_ms": 42.168
  },
  {
    "request_id": 17,
    "latency_ms": 45.795
  },
  {
    "request_id": 18,
    "latency_ms": 43.048
  },
  {
    "request_id": 19,
    "latency_ms": 45.384
  }
]
============================================================

============================================================
STATISTICAL VALIDATION RESULTS:
============================================================
{
  "count": 20,
  "sum": 895.029,
  "mean": 44.7514,
  "min": 41.741,
  "max": 48.349
}

╔══════════════════════════════════════════════════════════════════════════════╗
║                    LATENCY BENCHMARK PERFORMANCE REPORT                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  TEST CONFIGURATION                                                           ║
╞══════════════════════════════════════════════════════════════════════════════╡
║  • Total Requests:    20                                          ║
║  • Baseline:           45.00 ms                                      ║
║  • Expected Std:        2.00 ms                                          ║
╟────────────────────────────────────────────────────────────────────────────────╢
║  LATENCY STATISTICS (latency_ms metric)                                       ║
╞══════════════════════════════════════════════════════════════════════════════╡
║  • Mean:             44.7514 ms                                      ║
║  • Median:           44.9790 ms                                     ║
║  • Min:              41.7410 ms                                      ║
║  • Max:              48.3490 ms                                      ║
║  • Range:             6.6080 ms                                     ║
║  • Std Deviation:     1.8314 ms                                   ║
╟────────────────────────────────────────────────────────────────────────────────╢
║  STATISTICAL VALIDATION (run_deterministic_stat_check)                         ║
╞══════════════════════════════════════════════════════════════════════════════╡
║  {
  "count": 20,
  "sum": 895.029,
  "mean": 44.7514,
  "min": 41.741,
  "max": 48.349
}
╟────────────────────────────────────────────────────────────────────────────────╢
║  QUALITY METRICS                                                              ║
╞══════════════════════════════════════════════════════════════════════════════╡
║  • Mean vs Baseline:    -0.25 ms deviation       ║
║  • CV (Coeff. Var):   4.09%              ║
║  • Outliers (>2σ):      0 requests ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FINAL SUMMARY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The latency benchmark generated 20 synthetic samples with a mean of 
44.7514 ms (baseline: 45.00 ms). The statistical valida

[Step 1: Duration 57.62 seconds| Input tokens: 2,519 | Output tokens: 2,383]

... (18 outputs left out)

[Step 6: Duration 60.79 seconds| Input tokens: 73,826 | Output tokens: 10,199]

Reached max steps.

[Step 7: Duration 161.43 seconds| Input tokens: 98,165 | Output tokens: 13,826]

# Latency Benchmark Simulation - Complete Solution

## Architectural Decisions & Reasoning

### Data Generation Strategy
I'll use a **Gaussian distribution** with:
- **Mean**: 45.0 ms (baseline)
- **Standard Deviation**: ~2.0 ms (small variations for realism)

This models real-world API latency where responses cluster around a baseline with natural jitter.

### Data Structure
Each record will contain:
- `request_id`: Sequential identifier (0-19)
- `latency_ms`: The simulated latency value

### Statistical Validation
The `run_deterministic_stat_check` tool will validate:
- Mean latency (should be ~45.0 ms)
- Min/max bounds (should be within reasonable range)
- Distribution consistency

---

## Implementation

### Step 1: Generate Synthetic Latency Data

```python
import random
import math

# Configuration
NUM_SAMPLES = 20
BASELINE_LATENCY = 45.0
STDEV = 2.0  # Small variation for realistic latency

def generate_latency_sample(base: float, std: float, rng: random.Random) -> float:
    ""

In [5]:
# Create a sample raw log file locally to triage
Path("runtime_errors.log").write_text(
    """
2026-11-01 10:14:02 ERROR [DBPool] Connection reset by peer: 127.0.0.1:5432
2026-11-01 10:15:11 WARN  [DiskWorker] Cache partition utilization > 85%
2026-11-01 10:18:45 ERROR [AuthService] Token signature expired for uid_982
2026-11-01 10:20:00 ERROR [DBPool] Connection reset by peer: 127.0.0.1:5432
2026-11-01 10:22:15 ERROR [DBPool] Connection timeout after 3000ms
2026-11-01 10:25:30 INFO  [Worker] Batch job 402 completed
    """.strip()
)

triage_guidelines = """
You are an Site Reliability Engineering Lead.
- You operate with zero tolerance for guesswork: parse logs systematically.
- Categorize errors by subsystem, calculate their exact counts, and assess the root cause.
- When generating reports, save them directly to disk using `write_local_file`.
"""

triage_agent = create_specialized_agent(triage_guidelines)

task_1 = """
1. Read 'runtime_errors.log' using the file tool.
2. Filter for lines containing 'ERROR'.
3. Count occurrences per component/subsystem.
4. Save an incident summary markdown file to 'triage_report.md' detailing:
   - Total errors found
   - Breakdown per subsystem
   - High-level recommendation
5. Return the report text as your final answer.
"""

print("\n--- Running Use Case 1: Log Triage ---")
result_1 = triage_agent.run(task_1)
print(result_1)


--- Running Use Case 1: Log Triage ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 1. Read 'runtime_errors.log' using the file tool.                                                               │
│ 2. Filter for lines containing 'ERROR'.                                                                         │
│ 3. Count occurrences per component/subsystem.                                                                   │
│ 4. Save an incident summary markdown file to 'triage_report.md' detailing:                                      │
│    - Total errors found                                                                                         │
│    - Breakdown per subsystem                                                                                    │
│    - High-level recommendation                                                                                  │
│ 5. Return the report text as your final answer.                                                                 │
│                                                                                                                 │
╰─ OpenAIModel - Qwen3.5-9B-Claude-4.6-HighIQ-INSTRUCT-HERETIC-UNCENSORED-mlx-4Bit ───────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  content = read_local_file(file_path="runtime_errors.log", max_chars=1000000)                                     
  print(f"File read successfully. Length: {len(content)} characters")                                              
  print("First 500 characters (preview):")                                                                         
  print(content[:500])                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
File read successfully. Length: 426 characters
First 500 characters (preview):
2026-11-01 10:14:02 ERROR [DBPool] Connection reset by peer: 127.0.0.1:5432
2026-11-01 10:15:11 WARN  [DiskWorker] Cache partition utilization > 85%
2026-11-01 10:18:45 ERROR [AuthService] Token signature expired for uid_982
2026-11-01 10:20:00 ERROR [DBPool] Connection reset by peer: 127.0.0.1:5432
2026-11-01 10:22:15 ERROR [DBPool] Connection timeout after 3000ms
2026-11-01 10:25:30 INFO  [Worker] Batch job 402 completed

Out: None

[Step 1: Duration 3.59 seconds| Input tokens: 2,499 | Output tokens: 107]

... (18 outputs left out)

[Step 6: Duration 8.81 seconds| Input tokens: 24,699 | Output tokens: 1,622]

Reached max steps.

[Step 7: Duration 27.81 seconds| Input tokens: 29,367 | Output tokens: 2,371]

<code>
# Read the runtime_errors.log file
content = read_local_file(file_path="runtime_errors.log", max_chars=1000000)

# Filter for ERROR lines and extract component names
error_lines = []
components = {}

for line in content.split('\n'):
    if 'ERROR' in line:
        error_lines.append(line)
        # Extract component from format: YYYY-MM-DD HH:MM:SS LEVEL [Component] Message
        if '[' in line and ']' in line:
            component = line.split('[')[1].split(']')[0]
            components[component] = components.get(component, 0) + 1

total_errors = len(error_lines)

# Generate the incident summary markdown report
report = f"""# Runtime Error Triage Report

## Executive Summary
- **Total Errors Found**: {total_errors}
- **Analysis Date**: 2026-11-01
- **Affected Subsystems**: {len(components)}

---

## Error Breakdown by Subsystem

| Subsystem | Error Count |
|-----------|-------------|
"""

for component, count in sorted(components.items()):
    report += f"| {component} | {